# Track identification: relation-aware r-GCN/HGT with invariant RAO and variable radar mode

This notebook is derived from `observation_series_and_intel_rgcn_classification_advanced_network_llm.ipynb`. It changes the learning unit from an individual observation to a hierarchical track/observation representation:

* the r-GCN/HGT encoder still produces contextual observation embeddings;
* a gated, mask-aware pool produces exactly one embedding per `series_id`;
* one joint, KG-valid aircraft/radar/operator (**RAO**) head classifies each track;
* a radar-mode head classifies each observation, conditioned on its local embedding and the soft track-level RAO distribution; and
* categorical and Dirichlet/Subjective-Logic losses are balanced by track, so long series do not dominate.

The output schema, evaluation dashboard, and local-Ollama explanation likewise contain one invariant RAO assessment per track and an ordered, changeable radar-mode sequence. Ground truth remains outside inference features, and train/test/validation allocation remains grouped by complete tracks.


In [ ]:
from __future__ import annotations

import json, math, os, random, sys, time
import urllib.error, urllib.parse, urllib.request
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.utils.checkpoint
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rgcn_fusion import attribute_assessments, decode_kg_constrained
from esm_observation_series_generator import load_observation_series_json
from rgcn_fusion.intelligence_reports import (
    aggregate_candidate_intelligence,
    report_claim_score,
    report_observation_proximity,
    report_recency_score,
)
from kg_generator import generate_graph
from rgcn_fusion.observation_etl import ds_masses_from_score, score_candidates
DATA_PATH = ROOT / "generated" / "demo_esm_observation_series_with_sightings_and_patterns.json"
ARTIFACT_DIR = ROOT / "artifacts" / "Track_identification"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_PATH)
print(ARTIFACT_DIR)
print(DEVICE)



## Load series and create a label-free inference view

Ground-truth fields are retained in a separate target table, but are recursively stripped from the feature payload before graph construction. This prevents leakage from `ground_truth_label`, `ground_truth_track_label`, and `ground_truth_mode_sequence` into node features or edges.



In [ ]:
LEAKAGE_KEYS = {"ground_truth_label", "ground_truth_track_label", "ground_truth_mode_sequence", "synthetic_truth_value"}
TARGETS = ["aircraft_variant", "radar_mode", "radar_type", "operator_country"]

def strip_ground_truth(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {k: strip_ground_truth(v) for k, v in obj.items() if k not in LEAKAGE_KEYS}
    if isinstance(obj, list):
        return [strip_ground_truth(v) for v in obj]
    return obj

raw = load_observation_series_json(DATA_PATH)
series_records = raw["observation_series"]
inference_records = strip_ground_truth(series_records)

# Evaluation labels remain outside the inference graph. Track-level truth is retained
# separately so the final explanation output can show the correct identification.
track_targets_by_series_id = {
    series["series_id"]: dict(series["ground_truth_track_label"])
    for series in series_records
}
if len(track_targets_by_series_id) != len(series_records):
    raise ValueError("series_id values must be unique to report correct track identifications")

# Observation targets are extracted only from the original payload and never merged back into features.
target_rows = []
for s in series_records:
    for obs in s["observations"]:
        gt = obs["ground_truth_label"]
        target_rows.append({
            "observation_id": obs["observation_id"],
            "series_id": obs["series_id"],
            "sequence_index": obs["sequence_index"],
            "aircraft_variant": gt.get("aircraft_variant"),
            "radar_mode": gt.get("mode") or gt.get("radar_mode"),
            "radar_type": gt.get("radar") or gt.get("radar_type"),
            "operator_country": gt.get("operator") or gt.get("operator_country"),
        })

serialized_features = json.dumps(inference_records)
assert "ground_truth" not in serialized_features, "Ground-truth fields leaked into inference view"
report_count = sum(len(series.get("intelligence_reports") or []) for series in series_records)
print(f"series={len(series_records):,}, observations={len(target_rows):,}, intelligence_reports={report_count:,}")
print(target_rows[0])



## Feature engineering with proximity-linked intelligence and KG entities

Observation features use measured ESM, kinematics, time, and location. Sighting reports are connected only where their reported time and coordinates are close to an observation; pattern-of-life reports use collection time and expected operating area. Structured aircraft, family, radar, and operator claims also connect directly to the corresponding existing KG entity nodes, allowing the NN to propagate report evidence through the domain graph.


In [ ]:
INCLUDE_CANDIDATE_NODES = True
INCLUDE_INTEL_REPORT_NODES = True
USE_SEGMENT_EDGES = False
SEGMENT_FREQUENCY_SHIFT_GHZ = 0.75
MAX_KG_CANDIDATES = 5
GRAPH_INPUT_ARTIFACT = ARTIFACT_DIR / "graph_construction_outputs.pt"
X_NP_DTYPE, TORCH_DATA_DTYPE = np.float16, torch.float16

kg = generate_graph()
kg_nodes = {node["id"]: node for node in kg["nodes"]}
radar_by_mode = {edge["target"]: edge["source"] for edge in kg["edges"] if edge["relation"] == "HAS_MODE"}
aircraft_by_radar: dict[str, list[str]] = defaultdict(list)
operators_by_aircraft: dict[str, list[str]] = defaultdict(list)
aircraft_family_by_aircraft: dict[str, str] = {}
for edge in kg["edges"]:
    if edge["relation"] == "USES_RADAR":
        aircraft_by_radar[edge["target"]].append(edge["source"])
    elif edge["relation"] == "OPERATES":
        operators_by_aircraft[edge["target"]].append(kg_nodes[edge["source"]]["properties"].get("name"))
    elif edge["relation"] == "VARIANT_OF":
        aircraft_family_by_aircraft[edge["source"]] = edge["target"]
kg_candidate_rows = [
    {
        "mode_id": mode_id, "mode_props": kg_nodes[mode_id]["properties"],
        "radar_id": radar_id, "radar_props": kg_nodes[radar_id]["properties"],
        "aircraft_id": aircraft_id,
        "aircraft_props": kg_nodes[aircraft_id]["properties"] if aircraft_id else None,
        "aircraft_uses_radar": aircraft_id is not None, "operator": operator,
    }
    for mode_id, radar_id in radar_by_mode.items()
    for aircraft_id in (aircraft_by_radar.get(radar_id) or [None])
    for operator in (operators_by_aircraft.get(aircraft_id, [None]) if aircraft_id else [None])
]

# Elementary DS worlds are complete, mutually exclusive KG-valid configurations.
# Aircraft family is projected from the joint frame rather than trained as an
# unrelated head, preserving its deterministic relationship to the variant.
kg_valid_worlds = []
seen_kg_worlds = set()
for row in kg_candidate_rows:
    if not row["aircraft_id"] or not row["operator"]:
        continue
    world = {
        "radar_mode": row["mode_props"].get("name"),
        "radar_type": row["radar_props"].get("name"),
        "aircraft_variant": row["aircraft_props"].get("variant"),
        "aircraft_family": row["aircraft_props"].get("family"),
        "operator_country": row["operator"],
    }
    key = tuple(world[name] for name in ("radar_mode", "radar_type", "aircraft_variant", "aircraft_family", "operator_country"))
    if None not in key and key not in seen_kg_worlds:
        seen_kg_worlds.add(key)
        kg_valid_worlds.append(world)
print(f"KG-valid joint frame: {len(kg_valid_worlds):,} elementary worlds")

# score_candidates repeats the same mode/aircraft calculation once per operator.
# Score one template per mode/radar/aircraft tuple, then expand its operator variants.
candidate_variants: dict[tuple[str, str | None, str | None], list[dict[str, Any]]] = defaultdict(list)
for row in kg_candidate_rows:
    candidate_variants[(row["mode_id"], row["radar_id"], row["aircraft_id"])].append(row)
candidate_templates = [{**rows[0], "operator": None} for rows in candidate_variants.values()]


def flatten_numeric(prefix: str, value: Any, out: dict[str, float]) -> None:
    if isinstance(value, dict):
        for k, v in value.items():
            flatten_numeric(f"{prefix}.{k}" if prefix else k, v, out)
    elif isinstance(value, (int, float)) and not isinstance(value, bool):
        out[prefix] = float(value)


def _numeric_value(value: Any) -> float | None:
    if isinstance(value, dict):
        value = value.get("value")
    return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) else None


def segment_indices(observations: list[dict[str, Any]]) -> list[int]:
    """Infer coarse transition markers from measured frequency only, never labels/candidates."""
    frequencies = [_numeric_value(obs.get("esm_radar_parameters", {}).get("measured_centre_frequency_ghz")) for obs in observations]
    segments, current = [], 0
    for index, frequency in enumerate(frequencies):
        if index and frequency is not None and frequencies[index - 1] is not None and abs(frequency - frequencies[index - 1]) > SEGMENT_FREQUENCY_SHIFT_GHZ:
            current += 1
        segments.append(current)
    return segments


def candidate_scores_for_observation(obs: dict[str, Any]):
    """Avoid rescoring identical mode/radar/aircraft templates for every operator."""
    template_scores = score_candidates(obs, candidate_templates, max_candidates=len(candidate_templates))
    expanded = []
    context = obs.get("external_context") or {}
    priors = context.get("priors") if isinstance(context.get("priors"), dict) else {}
    operator_priors = context.get("operator_priors", priors.get("operator", {})) if isinstance(context, dict) else {}
    contextual_operator = context.get("operator") if isinstance(context, dict) else None
    for score in template_scores:
        for row in candidate_variants[(score.mode_id, score.radar_id, score.aircraft_id)]:
            operator = row["operator"]
            if isinstance(operator_priors, dict) and operator in operator_priors:
                operator_score = max(0.0, min(1.0, float(operator_priors[operator])))
            elif isinstance(contextual_operator, (list, tuple, set)):
                operator_score = 1.0 if operator in contextual_operator else 0.0
            elif contextual_operator is None:
                operator_score = 0.5
            else:
                operator_score = 1.0 if operator == contextual_operator else 0.0
            expanded.append((round(0.75 * score.mode_score + 0.15 * score.aircraft_score + 0.10 * operator_score, 6), score, operator))
    return sorted(expanded, key=lambda item: item[0], reverse=True)[:MAX_KG_CANDIDATES]



feature_rows, node_meta = [], []
observation_node_indices: list[int] = []
candidates_by_observation_node: dict[int, list[int]] = defaultdict(list)
reports_by_observation_node: dict[int, list[int]] = defaultdict(list)
claims_by_observation_node: dict[int, list[int]] = defaultdict(list)
claims_by_claim_type: dict[tuple[str, str], list[int]] = defaultdict(list)
claim_candidate_edges: list[dict[str, Any]] = []
kg_entity_node_indices: dict[str, int] = {}
report_kg_edges: list[tuple[int, int]] = []


In [ ]:

# Materialize the existing domain KG as NN nodes before adding track evidence.
# Claims refer to these nodes by stable KG id; no synthetic truth is needed.
for kg_node in kg["nodes"]:
    if kg_node.get("label") not in {"AircraftVariant", "AircraftFamily", "Radar", "Operator", "RadarMode"}:
        continue
    kg_idx = len(feature_rows)
    kg_entity_node_indices[kg_node["id"]] = kg_idx
    kg_features = {
        "node_kind_observation": 0.0, "node_kind_candidate": 0.0,
        "node_kind_report": 0.0, "node_kind_claim": 0.0, "node_kind_kg_entity": 1.0,
    }
    flatten_numeric("kg", kg_node.get("properties", {}), kg_features)
    feature_rows.append(kg_features)
    node_meta.append({"node_kind": "kg_entity", "kg_id": kg_node["id"], "kg_label": kg_node["label"]})

# Append every node in one traversal: this removes two large pending-node lists and
# parses each report timestamp exactly once, rather than once per claim.
for series in inference_records:
    obs_list = sorted(series["observations"], key=lambda obs: obs["sequence_index"])
    n = max(len(obs_list), 1)
    for obs, segment in zip(obs_list, segment_indices(obs_list)):
        obs_node_idx = len(feature_rows)
        observation_node_indices.append(obs_node_idx)
        feats = {
            "node_kind_observation": 1.0, "node_kind_candidate": 0.0,
            "elapsed_time_s": float(obs.get("elapsed_time_s", 0.0)),
            "sequence_fraction": float(obs.get("sequence_index", 0)) / max(n - 1, 1),
            "segment_index": float(segment), "duration_s": float(series.get("duration_s", 0.0)),
            "observation_count": float(series.get("observation_count", n)),
        }
        flatten_numeric("esm", obs.get("esm_radar_parameters", {}), feats)
        flatten_numeric("kin", obs.get("approximate_kinematics", {}), feats)
        flatten_numeric("loc", obs.get("estimated_emitter_location", {}), feats)
        feature_rows.append(feats)
        timestamp = obs.get("timestamp_iso8601")
        observation_time = datetime.fromisoformat(timestamp.replace("Z", "+00:00")).astimezone(UTC) if timestamp else None
        obs_meta = {"node_kind": "observation", "observation_id": obs["observation_id"], "series_id": obs["series_id"], "sequence_index": obs["sequence_index"], "timestamp_iso8601": timestamp}
        node_meta.append(obs_meta)

        if INCLUDE_INTEL_REPORT_NODES and obs is obs_list[0]:
            for report_rank, report in enumerate(series.get("intelligence_reports") or [], start=1):
                recency = report_recency_score(report, reference_time=observation_time)
                report_idx = len(feature_rows)
                feature_rows.append({"node_kind_observation": 0.0, "node_kind_candidate": 0.0, "node_kind_report": 1.0, "node_kind_claim": 0.0, "report_rank": float(report_rank), "report_credibility_score": float(report.get("credibility_score", 0.5)), "report_recency_score": float(recency), "report_claim_count": float(len(report.get("claims") or []))})
                node_meta.append({"node_kind": "intelligence_report", "report_id": report["report_id"], "report_type": report.get("report_type"), "report_payload": report, "observation_node_indices": [], "series_id": obs_meta["series_id"]})
                for claim_rank, claim in enumerate(report.get("claims") or [], start=1):
                    claim_idx = len(feature_rows)
                    claim_score = report_claim_score(report, claim, observation_time=observation_time)
                    feature_rows.append({"node_kind_observation": 0.0, "node_kind_candidate": 0.0, "node_kind_report": 0.0, "node_kind_claim": 1.0, "claim_rank": float(claim_rank), "claim_confidence": float(claim.get("claim_confidence", 0.5)), "claim_extraction_confidence": float(claim.get("extraction_confidence", 0.5)), "claim_specificity_score": float(claim.get("specificity_score", 0.5)), "claim_kg_consistency_score": float(claim.get("kg_consistency_score", 0.5)), "claim_text_score": float(claim_score), "claim_supports": 1.0 if claim.get("stance", "supports") == "supports" else 0.0, "claim_refutes": 1.0 if claim.get("stance") == "refutes" else 0.0})
                    node_meta.append({"node_kind": "report_claim", "id": f"evidence:claim:{claim['claim_id']}", "claim_id": claim["claim_id"], "claim_type": claim.get("claim_type"), "object_id": claim.get("object_id"), "kg_entity_id": claim.get("kg_entity_id"), "stance": claim.get("stance", "supports"), "text_score": claim_score, "source_id": report.get("source_id"), "report_id": report.get("report_id"), "report_node_idx": report_idx, "observation_node_indices": [], "series_id": obs_meta["series_id"]})

        if INCLUDE_CANDIDATE_NODES:
            # Claims are scored at this observation time, then passed to the shared
            # production aggregation function. Compatibility is already final and
            # signed, so no additional stance multiplier is applied in the notebook.
            applicable_claims = []
            for report in series.get("intelligence_reports") or []:
                if report_observation_proximity(report, obs) is None:
                    continue
                for claim in report.get("claims") or []:
                    applicable_claims.append({
                        **claim,
                        "id": f"evidence:claim:{claim['claim_id']}",
                        "report_id": report.get("report_id"),
                        "source_id": report.get("source_id"),
                        "series_id": obs_meta["series_id"],
                        "text_score": report_claim_score(report, claim, observation_time=observation_time),
                    })

            enriched_candidates = []
            for sensor_rank, (sensor_score, score, operator) in enumerate(candidate_scores_for_observation(obs), start=1):
                candidate_id = f"candidate:{obs_meta['observation_id']}:{sensor_rank}"
                candidate = {
                    "id": candidate_id,
                    "observation_id": obs_meta["observation_id"],
                    "series_id": obs_meta["series_id"],
                    "mode_id": score.mode_id,
                    "radar_id": score.radar_id,
                    "aircraft_id": score.aircraft_id,
                    "aircraft_family_id": aircraft_family_by_aircraft.get(score.aircraft_id),
                    "operator": operator,
                    "relation_id": f"relation:{score.aircraft_id}:USES_RADAR:{score.radar_id}",
                    "sensor_score": sensor_score,
                    "sensor_ds_masses": ds_masses_from_score(sensor_score, 0.2 if sensor_rank == 1 else 0.35),
                }
                intelligence, direct_edges = aggregate_candidate_intelligence(candidate, applicable_claims)
                enriched_candidates.append((intelligence["final_score"], sensor_rank, score, operator, candidate, intelligence, direct_edges))

            enriched_candidates.sort(key=lambda item: item[0], reverse=True)
            candidate_count = len(enriched_candidates)
            for rank, (_final_score, sensor_rank, score, operator, candidate, intelligence, direct_edges) in enumerate(enriched_candidates, start=1):
                candidate_idx = len(feature_rows)
                intel_features = {
                    f"candidate_{name}": float(intelligence[name])
                    for name in (
                        "sensor_score", "intel_support_score", "intel_refute_score",
                        "intel_net_score", "intel_score", "intel_conflict",
                        "intel_uncertainty", "intel_claim_count", "intel_source_count",
                        "intel_effective_weight", "final_score",
                    )
                }
                fused_non_match, fused_match, fused_uncertain = intelligence["ds_masses"]
                feature_rows.append({"node_kind_observation": 0.0, "node_kind_candidate": 1.0, "candidate_rank": float(rank), "candidate_sensor_rank": float(sensor_rank), "candidate_rank_fraction": float(rank - 1) / max(candidate_count - 1, 1), "candidate_count": float(candidate_count), "candidate_mode_score": float(score.mode_score), "candidate_aircraft_score": float(score.aircraft_score), "candidate_total_score": float(intelligence["final_score"]), "candidate_fused_non_match_mass": float(fused_non_match), "candidate_fused_match_mass": float(fused_match), "candidate_fused_uncertain_mass": float(fused_uncertain), "candidate_matched_fields": float(score.matched_fields), "candidate_compared_fields": float(score.compared_fields), **intel_features, **{f"candidate_{name}": float(value) for name, value in (getattr(score, "feature_scores", None) or {}).items()}})
                node_meta.append({"node_kind": "candidate", "candidate_id": candidate["id"], "observation_node_idx": obs_node_idx, "observation_id": obs_meta["observation_id"], "series_id": obs_meta["series_id"], "sequence_index": obs_meta["sequence_index"], "rank": rank, "sensor_rank": sensor_rank, "mode_id": score.mode_id, "radar_id": score.radar_id, "aircraft_id": score.aircraft_id, "operator": operator, "sensor_score": intelligence["sensor_score"], "intel_score": intelligence["intel_score"], "intel_support_score": intelligence["intel_support_score"], "intel_refute_score": intelligence["intel_refute_score"], "intel_conflict": intelligence["intel_conflict"], "intel_uncertainty": intelligence["intel_uncertainty"], "final_score": intelligence["final_score"], "fused_ds_masses": intelligence["ds_masses"], "claim_evidence": direct_edges})
                candidates_by_observation_node[obs_node_idx].append(candidate_idx)
                claim_candidate_edges.extend({**edge, "candidate_node_idx": candidate_idx} for edge in direct_edges)

# Link reports by measured temporal/geographical proximity, not series membership alone.
observation_payload_by_id = {
    obs["observation_id"]: obs
    for series in inference_records
    for obs in series["observations"]
}
report_applicability: dict[int, list[int]] = {}
for report_idx, meta in enumerate(node_meta):
    if meta.get("node_kind") != "intelligence_report":
        continue
    applicable = []
    proximity_by_observation = {}
    for obs_idx in observation_node_indices:
        if node_meta[obs_idx]["series_id"] != meta["series_id"]:
            continue
        proximity = report_observation_proximity(
            meta["report_payload"], observation_payload_by_id[node_meta[obs_idx]["observation_id"]]
        )
        if proximity is not None:
            applicable.append(obs_idx)
            proximity_by_observation[obs_idx] = proximity
            reports_by_observation_node[obs_idx].append(report_idx)
    meta["observation_node_indices"] = applicable
    meta["proximity_by_observation"] = proximity_by_observation
    report_applicability[report_idx] = applicable

for claim_idx, meta in enumerate(node_meta):
    if meta.get("node_kind") != "report_claim":
        continue
    applicable = report_applicability.get(meta["report_node_idx"], [])
    meta["observation_node_indices"] = applicable
    for obs_idx in applicable:
        claims_by_observation_node[obs_idx].append(claim_idx)
        claims_by_claim_type[(node_meta[obs_idx]["observation_id"], str(meta.get("claim_type")))].append(claim_idx)
    kg_idx = kg_entity_node_indices.get(meta.get("kg_entity_id"))
    if kg_idx is not None:
        report_kg_edges.append((claim_idx, kg_idx))

feature_names = sorted({key for row in feature_rows for key in row})
label_like_feature_names = [name for name in feature_names if any(token in name.lower() for token in ("ground_truth", "mode_id", "radar_mode", "aircraft_id", "operator_country", "synthetic_truth"))]
assert not label_like_feature_names, f"Label-like feature names leaked: {label_like_feature_names}"

X_np = np.asarray([[row.get(name, 0.0) for name in feature_names] for row in feature_rows], dtype=X_NP_DTYPE)
mu, sigma = X_np.astype(np.float32).mean(axis=0), X_np.astype(np.float32).std(axis=0)
sigma[sigma == 0] = 1.0
X = torch.tensor((X_np.astype(np.float32) - mu) / sigma, dtype=TORCH_DATA_DTYPE, device=DEVICE)

torch.save({"version": 3, "X": X.cpu(), "feature_names": feature_names, "feature_rows": feature_rows, "node_meta": node_meta, "observation_node_indices": observation_node_indices, "candidates_by_observation_node": dict(candidates_by_observation_node), "reports_by_observation_node": dict(reports_by_observation_node), "claims_by_observation_node": dict(claims_by_observation_node), "claims_by_claim_type": dict(claims_by_claim_type), "claim_candidate_edges": claim_candidate_edges, "kg_entity_node_indices": kg_entity_node_indices, "report_kg_edges": report_kg_edges, "include_candidate_nodes": INCLUDE_CANDIDATE_NODES, "include_intel_report_nodes": INCLUDE_INTEL_REPORT_NODES, "use_segment_edges": USE_SEGMENT_EDGES, "segment_frequency_shift_ghz": SEGMENT_FREQUENCY_SHIFT_GHZ, "max_kg_candidates": MAX_KG_CANDIDATES}, GRAPH_INPUT_ARTIFACT)
print(X.shape, feature_names[:10], {"observation_nodes": len(observation_node_indices), "candidate_nodes": sum(map(len, candidates_by_observation_node.values())), "report_nodes": sum(1 for meta in node_meta if meta.get("node_kind") == "intelligence_report"), "claim_nodes": sum(1 for meta in node_meta if meta.get("node_kind") == "report_claim"), "artifact": str(GRAPH_INPUT_ARTIFACT)})


In [ ]:
GRAPH_INPUT_ARTIFACT = ARTIFACT_DIR / "graph_construction_outputs.pt"

torch.save({"version": 3, "X": X.cpu(), "feature_names": feature_names, "feature_rows": feature_rows, "node_meta": node_meta, "observation_node_indices": observation_node_indices, "candidates_by_observation_node": dict(candidates_by_observation_node), "reports_by_observation_node": dict(reports_by_observation_node), "claims_by_observation_node": dict(claims_by_observation_node), "claims_by_claim_type": dict(claims_by_claim_type), "claim_candidate_edges": claim_candidate_edges, "kg_entity_node_indices": kg_entity_node_indices, "report_kg_edges": report_kg_edges, "include_candidate_nodes": INCLUDE_CANDIDATE_NODES, "include_intel_report_nodes": INCLUDE_INTEL_REPORT_NODES, "use_segment_edges": USE_SEGMENT_EDGES, "segment_frequency_shift_ghz": SEGMENT_FREQUENCY_SHIFT_GHZ, "max_kg_candidates": MAX_KG_CANDIDATES}, GRAPH_INPUT_ARTIFACT)
print(X.shape, feature_names[:10], {"observation_nodes": len(observation_node_indices), "candidate_nodes": sum(map(len, candidates_by_observation_node.values())), "report_nodes": sum(1 for meta in node_meta if meta.get("node_kind") == "intelligence_report"), "claim_nodes": sum(1 for meta in node_meta if meta.get("node_kind") == "report_claim"), "claim_candidate_edges": len(claim_candidate_edges), "kg_entity_nodes": len(kg_entity_node_indices), "report_kg_edges": len(report_kg_edges), "artifact": str(GRAPH_INPUT_ARTIFACT)})


In [ ]:
# Reload graph-construction outputs instead of rebuilding the feature and evidence nodes.
GRAPH_INPUT_ARTIFACT = ARTIFACT_DIR / "graph_construction_outputs.pt"
graph_outputs = torch.load(GRAPH_INPUT_ARTIFACT, map_location="cpu", weights_only=False)
if graph_outputs.get("version") != 3:
    raise ValueError(f"Unsupported graph-construction artifact: {GRAPH_INPUT_ARTIFACT}")
X = graph_outputs["X"].to(device=DEVICE, dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32)
feature_names = graph_outputs["feature_names"]
feature_rows = graph_outputs["feature_rows"]
node_meta = graph_outputs["node_meta"]
observation_node_indices = graph_outputs["observation_node_indices"]
candidates_by_observation_node = defaultdict(list, graph_outputs["candidates_by_observation_node"])
reports_by_observation_node = defaultdict(list, graph_outputs["reports_by_observation_node"])
claims_by_observation_node = defaultdict(list, graph_outputs["claims_by_observation_node"])
claims_by_claim_type = defaultdict(list, graph_outputs["claims_by_claim_type"])
claim_candidate_edges = graph_outputs["claim_candidate_edges"]
kg_entity_node_indices = graph_outputs["kg_entity_node_indices"]
report_kg_edges = graph_outputs["report_kg_edges"]
print(f"Reloaded {X.size(0):,} nodes, {X.size(1):,} features from {GRAPH_INPUT_ARTIFACT}")


## Build the proximity- and KG-aware relational graph

The graph combines temporal track structure, candidates, report provenance, proximity-qualified report/claim links, and direct bidirectional claim-to-KG-entity links. The r-GCN/HGT can therefore distinguish report proximity, asserted domain identity, candidate support, and candidate refutation as separate relations.


In [ ]:
RELATION_NAMES = ["self", "next_observation", "prev_observation", "same_emitter"]
if USE_SEGMENT_EDGES:
    RELATION_NAMES.extend(["same_mode_segment", "possible_mode_shift"])
if INCLUDE_CANDIDATE_NODES:
    RELATION_NAMES.extend(["has_candidate", "candidate_for", "contradicts_candidate"])
if INCLUDE_INTEL_REPORT_NODES:
    RELATION_NAMES.extend(["has_report", "report_for", "report_contains_claim", "claim_from_report", "claim_supports_observation", "observation_supported_by_claim", "claim_supports_candidate", "candidate_supported_by_claim", "claim_refutes_candidate", "candidate_refuted_by_claim", "contradicts_claim", "claim_asserts_kg_entity", "kg_entity_asserted_by_claim"])
RELATIONS = {name: idx for idx, name in enumerate(RELATION_NAMES)}
edge_src, edge_dst, edge_type = [], [], []


def add_edge(i, j, rel):
    edge_src.append(i)
    edge_dst.append(j)
    edge_type.append(RELATIONS[rel])


def candidate_contradiction_reasons(left: dict[str, Any], right: dict[str, Any]) -> list[str]:
    comparisons = {"rank": (left.get("rank"), right.get("rank"))}
    return [field for field, (a, b) in comparisons.items() if a is not None and b is not None and a != b]


for i in range(len(node_meta)):
    add_edge(i, i, "self")

series_to_indices = defaultdict(list)
for i in observation_node_indices:
    series_to_indices[node_meta[i]["series_id"]].append(i)

segment_by_node = {i: int(feature_rows[i]["segment_index"]) for i in observation_node_indices}
for indices in series_to_indices.values():
    indices = sorted(indices, key=lambda i: node_meta[i]["sequence_index"])
    for a, b in zip(indices, indices[1:]):
        add_edge(a, b, "next_observation")
        add_edge(b, a, "prev_observation")
        if USE_SEGMENT_EDGES:
            if segment_by_node[a] == segment_by_node[b]:
                add_edge(a, b, "same_mode_segment")
                add_edge(b, a, "same_mode_segment")
            else:
                add_edge(a, b, "possible_mode_shift")
                add_edge(b, a, "possible_mode_shift")
    # Sparse same-emitter reinforcement edges between observations two samples apart.
    for a, b in zip(indices, indices[2:]):
        add_edge(a, b, "same_emitter")
        add_edge(b, a, "same_emitter")

contradiction_reasons = Counter()
if INCLUDE_CANDIDATE_NODES:
    for obs_idx, candidate_indices in candidates_by_observation_node.items():
        candidate_indices = sorted(candidate_indices, key=lambda idx: node_meta[idx]["rank"])
        for candidate_idx in candidate_indices:
            add_edge(obs_idx, candidate_idx, "has_candidate")
            add_edge(candidate_idx, obs_idx, "candidate_for")
        for left_pos, left_idx in enumerate(candidate_indices):
            for right_idx in candidate_indices[left_pos + 1:]:
                reasons = candidate_contradiction_reasons(node_meta[left_idx], node_meta[right_idx])
                if not reasons:
                    continue
                add_edge(left_idx, right_idx, "contradicts_candidate")
                contradiction_reasons.update(reasons)

if INCLUDE_INTEL_REPORT_NODES:
    for obs_idx, report_indices in reports_by_observation_node.items():
        for report_idx in report_indices:
            add_edge(obs_idx, report_idx, "has_report")
            add_edge(report_idx, obs_idx, "report_for")
    for obs_idx, claim_indices in claims_by_observation_node.items():
        for claim_idx in claim_indices:
            report_idx = node_meta[claim_idx]["report_node_idx"]
            add_edge(report_idx, claim_idx, "report_contains_claim")
            add_edge(claim_idx, report_idx, "claim_from_report")
            add_edge(claim_idx, obs_idx, "claim_supports_observation")
            add_edge(obs_idx, claim_idx, "observation_supported_by_claim")
    claim_node_by_id = {
        meta.get("id"): idx
        for idx, meta in enumerate(node_meta)
        if meta.get("node_kind") == "report_claim"
    }
    for direct_edge in claim_candidate_edges:
        claim_idx = claim_node_by_id.get(direct_edge["source"])
        candidate_idx = direct_edge["candidate_node_idx"]
        if claim_idx is None:
            continue
        if direct_edge["contribution"] > 0.0:
            add_edge(claim_idx, candidate_idx, "claim_supports_candidate")
            add_edge(candidate_idx, claim_idx, "candidate_supported_by_claim")
        elif direct_edge["contribution"] < 0.0:
            add_edge(claim_idx, candidate_idx, "claim_refutes_candidate")
            add_edge(candidate_idx, claim_idx, "candidate_refuted_by_claim")
    for claim_idx, kg_idx in report_kg_edges:
        add_edge(claim_idx, kg_idx, "claim_asserts_kg_entity")
        add_edge(kg_idx, claim_idx, "kg_entity_asserted_by_claim")
    for claim_indices in claims_by_claim_type.values():
        for left_pos, left_idx in enumerate(claim_indices):
            for right_idx in claim_indices[left_pos + 1:]:
                if node_meta[left_idx].get("object_id") != node_meta[right_idx].get("object_id"):
                    add_edge(left_idx, right_idx, "contradicts_claim")
                    contradiction_reasons.update(["report_claim_object_id"])

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long, device=DEVICE)
edge_types = torch.tensor(edge_type, dtype=torch.long, device=DEVICE)
print({name: int((edge_types == rid).sum().cpu()) for name, rid in RELATIONS.items()})
print({"contradiction_reasons": dict(contradiction_reasons)})




## Build track-level RAO and observation-level radar-mode targets

The invariant label is one joint `(aircraft_variant, radar_type, operator_country)` class per series. Radar mode remains an observation target. The 0.5/0.3/0.2 split is performed on `series_id` and stratified by the joint RAO class; no observation from a track can cross a split boundary.


In [ ]:
RAO_FIELDS = ("aircraft_variant", "radar_type", "operator_country")
MODE_TASK = "radar_mode"

def rao_tuple(values: dict[str, Any]) -> tuple[str, str, str]:
    """Normalize either track- or observation-label naming to one RAO tuple."""
    normalized = {
        "aircraft_variant": values.get("aircraft_variant") or values.get("aircraft"),
        "radar_type": values.get("radar_type") or values.get("radar"),
        "operator_country": values.get("operator_country") or values.get("operator"),
    }
    result = tuple(normalized[field] for field in RAO_FIELDS)
    if any(value is None for value in result):
        raise ValueError(f"Incomplete RAO target: {values}")
    return result

# Validate the invariant before training rather than silently optimizing contradictory labels.
rao_target_by_series = {
    series_id: rao_tuple(target) for series_id, target in track_targets_by_series_id.items()
}
observation_rows_by_series: dict[str, list[dict[str, Any]]] = defaultdict(list)
for row in target_rows:
    observation_rows_by_series[row["series_id"]].append(row)
for series_id, rows in observation_rows_by_series.items():
    mismatches = [row["observation_id"] for row in rows if rao_tuple(row) != rao_target_by_series[series_id]]
    if mismatches:
        raise ValueError(f"RAO changes inside {series_id!r}; split the track at {mismatches[:3]}")

# The RAO output vocabulary is an allow-list of KG-valid worlds, not a Cartesian product.
rao_vocab = sorted({
    tuple(world[field] for field in RAO_FIELDS)
    for world in kg_valid_worlds
})
rao_class_index = {world: index for index, world in enumerate(rao_vocab)}
unknown_targets = {value for value in rao_target_by_series.values() if value not in rao_class_index}
if unknown_targets:
    raise ValueError(f"Track targets absent from the KG-valid RAO frame: {sorted(unknown_targets)[:3]}")

mode_vocab = sorted({row[MODE_TASK] for row in target_rows})
if any(value is None for value in mode_vocab):
    raise ValueError("Missing radar-mode labels are not supported")
mode_class_index = {value: index for index, value in enumerate(mode_vocab)}

series_ids = sorted(rao_target_by_series)
series_index = {series_id: index for index, series_id in enumerate(series_ids)}
track_targets = torch.tensor(
    [rao_class_index[rao_target_by_series[series_id]] for series_id in series_ids],
    dtype=torch.long, device=DEVICE,
)
target_by_observation_id = {row["observation_id"]: row for row in target_rows}
observation_nodes = torch.tensor(observation_node_indices, dtype=torch.long, device=DEVICE)
observation_track_index = torch.tensor(
    [series_index[node_meta[node_idx]["series_id"]] for node_idx in observation_node_indices],
    dtype=torch.long, device=DEVICE,
)
mode_targets = torch.tensor(
    [mode_class_index[target_by_observation_id[node_meta[node_idx]["observation_id"]][MODE_TASK]] for node_idx in observation_node_indices],
    dtype=torch.long, device=DEVICE,
)

# Compatibility[RAO, mode] is derived from the KG and is used by the conditioned mode head.
rao_mode_compatibility = torch.zeros((len(rao_vocab), len(mode_vocab)), dtype=torch.float32, device=DEVICE)
for world in kg_valid_worlds:
    rao = tuple(world[field] for field in RAO_FIELDS)
    mode = world[MODE_TASK]
    if rao in rao_class_index and mode in mode_class_index:
        rao_mode_compatibility[rao_class_index[rao], mode_class_index[mode]] = 1.0
if (rao_mode_compatibility.sum(dim=1) == 0).any():
    raise ValueError("Every RAO class must support at least one radar mode")


def stratified_track_split(labels_by_series, fractions, seed):
    split_names = tuple(fractions)
    desired = {name: fractions[name] * len(labels_by_series) for name in split_names}
    result = {name: set() for name in split_names}
    strata = defaultdict(list)
    for series_id, label in labels_by_series.items():
        strata[label].append(series_id)
    rng = np.random.default_rng(seed)
    items = list(strata.items())
    rng.shuffle(items)
    items.sort(key=lambda item: len(item[1]), reverse=True)
    for _, ids in items:
        rng.shuffle(ids)
        quota = {name: fractions[name] * len(ids) for name in split_names}
        assigned = {name: 0 for name in split_names}
        for series_id in ids:
            name = max(split_names, key=lambda key: (quota[key] - assigned[key], desired[key] - len(result[key])))
            result[name].add(series_id)
            assigned[name] += 1
    return result

split_series = stratified_track_split(rao_target_by_series, {"train": 0.5, "test": 0.3, "val": 0.2}, SEED)
track_splits = {
    name: torch.tensor([series_index[sid] for sid in series_ids if sid in ids], dtype=torch.long, device=DEVICE)
    for name, ids in split_series.items()
}
observation_splits = {
    name: torch.nonzero(torch.isin(observation_track_index, track_indices), as_tuple=False).flatten()
    for name, track_indices in track_splits.items()
}
for name in track_splits:
    print(name, {"tracks": int(track_splits[name].numel()), "observations": int(observation_splits[name].numel())})
print({"rao_classes": len(rao_vocab), "mode_classes": len(mode_vocab)})


## Hierarchical relation-aware encoder, track pool, and constrained output heads

The graph encoder is retained, but its observation embeddings are pooled by `series_id`. `TrackAttentionPool` returns one track vector. A single joint RAO head operates on those vectors, making within-track identity changes impossible by construction. The radar-mode head operates on each observation and receives its broadcast track vector plus a differentiable KG compatibility prior marginalized over the soft RAO distribution.


In [ ]:

MAXIMUM_HIDDEN_DIM = 120
MINIMUM_HIDDEN_DIM = 32
NUM_MESSAGE_PASSING_EDGES = 15  # user-configurable r-GCN depth
NUM_RGCN_BASES = 6  # basis decomposition bounds relation-specific parameter growth
NUM_HGT_LAYERS = 1
NUM_ATTENTION_HEADS = 4
DROPOUT = 0.3
TASK_HEAD_HIDDEN_DIM = 72
EVIDENTIAL_LOSS_WEIGHT = 0.8  # jointly train per-task Dirichlet/DS heads
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
L1_LAMBDA = 1e-5
PATIENCE = 4
EARLY_STOPPING_MIN_DELTA = 1e-4
EDGE_CHUNK_SIZE = 750_000

MODEL_DTYPE = torch.float32
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16
GRADIENT_CHECKPOINTING = DEVICE.type == "cuda"

if NUM_MESSAGE_PASSING_EDGES < 1:
    raise ValueError("NUM_MESSAGE_PASSING_EDGES must be at least 1.")
if MINIMUM_HIDDEN_DIM < 1 or MAXIMUM_HIDDEN_DIM < MINIMUM_HIDDEN_DIM:
    raise ValueError("Hidden dimensions must satisfy 1 <= MINIMUM_HIDDEN_DIM <= MAXIMUM_HIDDEN_DIM.")
if MINIMUM_HIDDEN_DIM % NUM_ATTENTION_HEADS != 0:
    raise ValueError("MINIMUM_HIDDEN_DIM must be divisible by NUM_ATTENTION_HEADS for HGT attention.")


class RelationalGraphConvolutionBlock(nn.Module):
    """Residual r-GCN aggregation with basis-decomposed relation weights."""
    def __init__(self, in_dim, out_dim, num_relations, num_bases, dropout, edge_chunk_size=0):
        super().__init__()
        self.num_relations = max(int(num_relations), 1)
        self.num_bases = min(max(int(num_bases), 1), self.num_relations)
        self.bases = nn.Parameter(torch.empty(self.num_bases, in_dim, out_dim))
        self.coefficients = nn.Parameter(torch.empty(self.num_relations, self.num_bases))
        self.root = nn.Linear(in_dim, out_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_dim))
        self.residual = nn.Identity() if in_dim == out_dim else nn.Linear(in_dim, out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(dropout)
        if edge_chunk_size < 0:
            raise ValueError("edge_chunk_size must be non-negative")
        self.edge_chunk_size = int(edge_chunk_size)
        nn.init.xavier_uniform_(self.bases)
        nn.init.xavier_uniform_(self.coefficients)

    def forward(self, x, edge_index, edge_types):
        if edge_index.numel() == 0:
            return self.norm(self.residual(x) + self.dropout(F.gelu(self.root(x) + self.bias)))
        src, dst = edge_index
        messages = x.new_zeros((x.size(0), self.bases.size(-1)))
        degree = x.new_zeros(x.size(0))
        relation_weights = torch.einsum("rb,bio->rio", self.coefficients, self.bases)
        for relation_id in range(self.num_relations):
            mask = edge_types == relation_id
            if not torch.any(mask):
                continue
            relation_src, relation_dst = src[mask], dst[mask]
            chunk_size = self.edge_chunk_size or int(relation_src.numel())
            for chunk_start in range(0, int(relation_src.numel()), chunk_size):
                chunk_stop = chunk_start + chunk_size
                chunk_src = relation_src[chunk_start:chunk_stop]
                chunk_dst = relation_dst[chunk_start:chunk_stop]
                transformed = x[chunk_src] @ relation_weights[relation_id]
                messages.index_add_(0, chunk_dst, transformed.to(messages.dtype))
                degree.index_add_(0, chunk_dst, torch.ones(chunk_dst.numel(), device=x.device, dtype=x.dtype))
        messages = messages / degree.clamp_min(1).unsqueeze(-1)
        updated = self.dropout(F.gelu(self.root(x) + messages + self.bias))
        return self.norm(self.residual(x) + updated)

class RelationAwareHGTLayer(nn.Module):
    """HGT-style relation-aware multi-head attention over typed graph edges."""
    def __init__(self, hidden_dim: int, num_relations: int, num_heads: int, dropout: float, edge_chunk_size: int = 0):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")
        self.num_relations = max(num_relations, 1)
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(hidden_dim, hidden_dim)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.relation_key = nn.Parameter(torch.empty(self.num_relations, num_heads, self.head_dim, self.head_dim))
        self.relation_value = nn.Parameter(torch.empty(self.num_relations, num_heads, self.head_dim, self.head_dim))
        self.relation_priority = nn.Parameter(torch.ones(self.num_relations, num_heads))
        self.output = nn.Linear(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        if edge_chunk_size < 0:
            raise ValueError("edge_chunk_size must be non-negative")
        self.edge_chunk_size = int(edge_chunk_size)
        nn.init.xavier_uniform_(self.relation_key)
        nn.init.xavier_uniform_(self.relation_value)

    def _edge_messages(self, x, src, dst, rel):
        q = self.query(x[dst]).view(-1, self.num_heads, self.head_dim)
        k = self.key(x[src]).view(-1, self.num_heads, self.head_dim)
        v = self.value(x[src]).view(-1, self.num_heads, self.head_dim)
        rel_k = self.relation_key[rel]
        rel_v = self.relation_value[rel]
        k = torch.einsum("ehd,ehdf->ehf", k, rel_k)
        v = torch.einsum("ehd,ehdf->ehf", v, rel_v)
        scores = (q * k).sum(dim=-1) / math.sqrt(self.head_dim)
        scores = scores * self.relation_priority[rel]
        # Per-destination sigmoid gates keep memory bounded without requiring dense per-node softmax buffers.
        weights = torch.sigmoid(scores).unsqueeze(-1)
        return (weights * v).reshape(src.numel(), -1)

    def forward(self, x, edge_index, edge_types):
        if edge_index.numel() == 0:
            return x
        src, dst = edge_index
        messages = torch.zeros_like(x)
        degree = torch.zeros(x.size(0), device=x.device, dtype=x.dtype)
        for rel_id in range(self.num_relations):
            mask = edge_types == rel_id
            if not torch.any(mask):
                continue
            rel_src, rel_dst = src[mask], dst[mask]
            if self.edge_chunk_size > 0:
                for start in range(0, int(rel_src.numel()), self.edge_chunk_size):
                    stop = start + self.edge_chunk_size
                    chunk_src, chunk_dst = rel_src[start:stop], rel_dst[start:stop]
                    chunk_rel = torch.full_like(chunk_src, rel_id)
                    messages.index_add_(0, chunk_dst, self._edge_messages(x, chunk_src, chunk_dst, chunk_rel).to(messages.dtype))
                    degree.index_add_(0, chunk_dst, torch.ones(chunk_dst.numel(), device=x.device, dtype=x.dtype))
            else:
                rel_tensor = torch.full_like(rel_src, rel_id)
                messages.index_add_(0, rel_dst, self._edge_messages(x, rel_src, rel_dst, rel_tensor).to(messages.dtype))
                degree.index_add_(0, rel_dst, torch.ones(rel_dst.numel(), device=x.device, dtype=x.dtype))
        messages = messages / degree.clamp_min(1).unsqueeze(-1)
        updated = self.output(messages)
        updated = self.dropout(F.gelu(updated))
        return self.norm(x + updated)

class TrackAttentionPool(nn.Module):
    """Gated attention normalized independently inside each track."""
    def __init__(self, hidden_dim, dropout):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.Tanh(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
        self.summary = nn.Sequential(nn.Linear(3 * hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, observation_embeddings, observation_to_track, num_tracks):
        pooled = []
        for track_index in range(num_tracks):
            members = observation_embeddings[observation_to_track == track_index]
            if members.numel() == 0:
                raise ValueError(f"Track {track_index} has no observation embeddings")
            weights = torch.softmax(self.score(members).squeeze(-1), dim=0)
            attentive = (weights.unsqueeze(-1) * members).sum(dim=0)
            pooled.append(self.norm(attentive + self.summary(torch.cat((attentive, members.mean(0), members.amax(0))))))
        return torch.stack(pooled)


class TrackRGCNHGTClassifier(nn.Module):
    """One KG-valid RAO opinion per track and one conditioned mode opinion per observation."""
    def __init__(self, in_dim, maximum_hidden_dim, minimum_hidden_dim, num_relations, num_rao_classes, num_mode_classes,
                 rao_mode_compatibility, num_message_passing_edges=7, num_rgcn_bases=4, num_hgt_layers=2,
                 num_heads=4, dropout=0.2, task_head_hidden_dim=128, edge_chunk_size=0, gradient_checkpointing=False):
        super().__init__()
        if num_message_passing_edges < 1:
            raise ValueError("num_message_passing_edges must be at least 1")
        span = maximum_hidden_dim - minimum_hidden_dim
        self.rgcn_hidden_dims = [maximum_hidden_dim] if num_message_passing_edges == 1 else [
            round(maximum_hidden_dim - span * i / (num_message_passing_edges - 1)) for i in range(num_message_passing_edges)
        ]
        self.input_projection = nn.Sequential(nn.Linear(in_dim, maximum_hidden_dim), nn.LayerNorm(maximum_hidden_dim), nn.GELU(), nn.Dropout(dropout))
        inputs = [maximum_hidden_dim, *self.rgcn_hidden_dims[:-1]]
        self.rgcn_layers = nn.ModuleList([
            RelationalGraphConvolutionBlock(a, b, num_relations, num_rgcn_bases, dropout, edge_chunk_size)
            for a, b in zip(inputs, self.rgcn_hidden_dims)
        ])
        hidden_dim = self.rgcn_hidden_dims[-1]
        self.hgt_layers = nn.ModuleList([RelationAwareHGTLayer(hidden_dim, num_relations, num_heads, dropout, edge_chunk_size) for _ in range(num_hgt_layers)])
        self.gradient_checkpointing = bool(gradient_checkpointing)
        self.track_pool = TrackAttentionPool(hidden_dim, dropout)
        def head(in_features, out_features):
            return nn.Sequential(nn.Linear(in_features, task_head_hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(task_head_hidden_dim, out_features))
        self.rao_head = head(hidden_dim, num_rao_classes)
        self.rao_evidential_head = head(hidden_dim, num_rao_classes)
        self.mode_head = head(2 * hidden_dim, num_mode_classes)
        self.mode_evidential_head = head(2 * hidden_dim, num_mode_classes)
        self.register_buffer("rao_mode_compatibility", rao_mode_compatibility.float())

    @staticmethod
    def dirichlet_ds_output(evidence_logits):
        evidence = F.softplus(evidence_logits)
        alpha = evidence + 1.0
        strength = alpha.sum(dim=-1, keepdim=True)
        return {"evidence": evidence, "alpha": alpha, "strength": strength, "probabilities": alpha / strength,
                "belief": evidence / strength, "uncertainty": evidence.size(-1) / strength}

    def encode(self, x, edge_index, edge_types):
        h = self.input_projection(x)
        for layer in self.rgcn_layers:
            h = torch.utils.checkpoint.checkpoint(layer, h, edge_index, edge_types, use_reentrant=False) if self.gradient_checkpointing and self.training else layer(h, edge_index, edge_types)
        for layer in self.hgt_layers:
            h = torch.utils.checkpoint.checkpoint(layer, h, edge_index, edge_types, use_reentrant=False) if self.gradient_checkpointing and self.training else layer(h, edge_index, edge_types)
        return h

    def forward(self, x, edge_index, edge_types, observation_nodes, observation_to_track, num_tracks):
        node_embeddings = self.encode(x, edge_index, edge_types)
        observation_embeddings = node_embeddings[observation_nodes]
        track_embeddings = self.track_pool(observation_embeddings, observation_to_track, num_tracks)
        rao_logits = self.rao_head(track_embeddings)
        rao_evidential = self.dirichlet_ds_output(self.rao_evidential_head(track_embeddings))
        mode_features = torch.cat((observation_embeddings, track_embeddings[observation_to_track]), dim=-1)
        # Soft conditioning preserves gradients through uncertain RAO alternatives.
        compatible_mode_probability = torch.softmax(rao_logits, dim=-1) @ self.rao_mode_compatibility
        compatibility_bias = compatible_mode_probability[observation_to_track].clamp_min(1e-8).log()
        mode_logits = self.mode_head(mode_features) + compatibility_bias
        mode_evidential_logits = self.mode_evidential_head(mode_features) + compatibility_bias
        return {
            "track_rao": rao_logits,
            "radar_mode": mode_logits,
            "evidential": {
                "track_rao": rao_evidential,
                "radar_mode": self.dirichlet_ds_output(mode_evidential_logits),
            },
            "track_embeddings": track_embeddings,
            "observation_embeddings": observation_embeddings,
        }

model_config = {
    "architecture": "hierarchical_track_rao_and_observation_mode",
    "maximum_hidden_dim": MAXIMUM_HIDDEN_DIM, "minimum_hidden_dim": MINIMUM_HIDDEN_DIM,
    "num_message_passing_edges": NUM_MESSAGE_PASSING_EDGES, "num_rgcn_bases": NUM_RGCN_BASES,
    "num_hgt_layers": NUM_HGT_LAYERS, "num_attention_heads": NUM_ATTENTION_HEADS,
    "dropout": DROPOUT, "task_head_hidden_dim": TASK_HEAD_HIDDEN_DIM,
    "rao_classes": len(rao_vocab), "radar_mode_classes": len(mode_vocab),
    "rao_invariance": "one pooled output per series_id",
    "mode_conditioning": "local observation + pooled track + soft KG compatibility",
}
model = TrackRGCNHGTClassifier(
    X.size(1), MAXIMUM_HIDDEN_DIM, MINIMUM_HIDDEN_DIM, len(RELATIONS), len(rao_vocab), len(mode_vocab), rao_mode_compatibility,
    num_message_passing_edges=NUM_MESSAGE_PASSING_EDGES, num_rgcn_bases=NUM_RGCN_BASES,
    num_hgt_layers=NUM_HGT_LAYERS, num_heads=NUM_ATTENTION_HEADS, dropout=DROPOUT,
    task_head_hidden_dim=TASK_HEAD_HIDDEN_DIM, edge_chunk_size=EDGE_CHUNK_SIZE,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
).to(device=DEVICE, dtype=MODEL_DTYPE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE.type == "cuda" and AMP_DTYPE == torch.float16)
print(model)
print(model_config)


In [ ]:
# Estimate edge-computation demand and EDGE_CHUNK_SIZE ceiling.
def estimate_edge_computation_requirements(edge_index, edge_types, relation_names, *, num_rgcn_layers, num_hgt_layers):
    edge_count = int(edge_index.size(1))
    relation_edge_counts = {name: int((edge_types == relation_id).sum().detach().cpu()) for name, relation_id in relation_names.items()}
    max_relation_edges = max(relation_edge_counts.values(), default=0)
    return {
        "node_count": int(edge_index.max().item() + 1) if edge_index.numel() else 0,
        "edge_count": edge_count, "num_rgcn_layers": int(num_rgcn_layers), "num_hgt_layers": int(num_hgt_layers),
        "max_relation_edges_per_layer": max_relation_edges,
        "maximum_required_edge_chunk_size": max_relation_edges,
        "total_edge_computations_per_forward": (num_rgcn_layers + num_hgt_layers) * edge_count,
        "note": "Every r-GCN and HGT layer processes the full typed graph once per forward pass.",
        "relation_edge_counts": relation_edge_counts,
    }

edge_compute_requirements = estimate_edge_computation_requirements(edge_index, edge_types, RELATIONS, num_rgcn_layers=NUM_MESSAGE_PASSING_EDGES, num_hgt_layers=NUM_HGT_LAYERS)
for key, value in edge_compute_requirements.items():
    if key != "relation_edge_counts":
        print(f"{key}: {value:,}" if isinstance(value, int) else f"{key}: {value}")
print("relation_edge_counts:")
for relation_name, count in edge_compute_requirements["relation_edge_counts"].items():
    print(f"  {relation_name}: {count:,}")


## Train once per full graph with track-balanced supervision

Each epoch performs one full relational forward pass. RAO loss is averaged over training tracks. Mode loss is first averaged within each track and then across tracks, preventing long series from dominating. Expected Dirichlet cross-entropy is supplemented by an annealed KL penalty on incorrect evidence. Checkpoint selection uses the hierarchical validation loss.


In [ ]:
# Conservative memory preflight for the hierarchical full-graph model.
ENFORCE_GPU_MEMORY_CHECK = True
GPU_MEMORY_SAFETY_FACTOR = 1.50

def estimate_peak_training_gpu_memory():
    parameter_bytes = sum(parameter.numel() * parameter.element_size() for parameter in model.parameters())
    activation_bytes = X.size(0) * max(model.rgcn_hidden_dims) * 4 * (len(model.rgcn_layers) + len(model.hgt_layers) + 2)
    estimate = int((parameter_bytes * 4 + activation_bytes) * GPU_MEMORY_SAFETY_FACTOR)
    if DEVICE.type != "cuda":
        return {"gpu_available": False, "estimated_peak_bytes": estimate, "training_can_start": True,
                "reason": "CUDA unavailable; CPU training remains permitted."}
    free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)
    sufficient = estimate <= free_bytes
    return {"gpu_available": True, "device": torch.cuda.get_device_name(DEVICE), "estimated_peak_bytes": estimate,
            "currently_free_bytes": free_bytes, "total_bytes": total_bytes, "sufficient": sufficient,
            "training_can_start": sufficient if ENFORCE_GPU_MEMORY_CHECK else True}

gpu_memory_preflight = estimate_peak_training_gpu_memory()
TRAINING_CAN_START = bool(gpu_memory_preflight["training_can_start"])
print(gpu_memory_preflight)
if not TRAINING_CAN_START:
    raise RuntimeError("GPU memory preflight blocked training; reduce graph/model size or disable the conservative check explicitly")


In [ ]:
from torch.utils.tensorboard import SummaryWriter

RAO_CLASSIFICATION_WEIGHT = 1.0
MODE_CLASSIFICATION_WEIGHT = 1.0
RAO_EVIDENTIAL_WEIGHT = 0.8
MODE_EVIDENTIAL_WEIGHT = 0.8
EVIDENTIAL_KL_WEIGHT = 0.05
EVIDENTIAL_KL_ANNEAL_EPOCHS = 20


def expected_dirichlet_ce(alpha, labels):
    return torch.digamma(alpha.sum(-1)) - torch.digamma(alpha.gather(1, labels[:, None]).squeeze(1))


def dirichlet_kl_to_uniform(alpha):
    classes = alpha.size(-1)
    sum_alpha = alpha.sum(-1)
    return (torch.lgamma(sum_alpha) - torch.lgamma(alpha).sum(-1) - torch.lgamma(alpha.new_tensor(float(classes)))
            + ((alpha - 1.0) * (torch.digamma(alpha) - torch.digamma(sum_alpha).unsqueeze(-1))).sum(-1))


def suppress_true_class_evidence(alpha, labels):
    mask = F.one_hot(labels, alpha.size(-1)).to(alpha.dtype)
    return alpha * (1.0 - mask) + mask


def mean_modes_per_track(values, observation_positions, selected_tracks):
    per_track = []
    selected_track_ids = observation_track_index[observation_positions]
    selected_values = values
    for track_index in selected_tracks:
        members = selected_values[selected_track_ids == track_index]
        if members.numel():
            per_track.append(members.mean())
    if not per_track:
        raise ValueError("A split contains no radar-mode observations")
    return torch.stack(per_track).mean()


def hierarchical_loss(outputs, split_name, kl_coefficient=0.0):
    tracks = track_splits[split_name]
    observations = observation_splits[split_name]
    rao_labels = track_targets[tracks]
    mode_labels = mode_targets[observations]
    rao_ce = F.cross_entropy(outputs["track_rao"][tracks], rao_labels)
    mode_ce_each = F.cross_entropy(outputs["radar_mode"][observations], mode_labels, reduction="none")
    mode_ce = mean_modes_per_track(mode_ce_each, observations, tracks)
    rao_alpha = outputs["evidential"]["track_rao"]["alpha"][tracks]
    mode_alpha = outputs["evidential"]["radar_mode"]["alpha"][observations]
    rao_edl = expected_dirichlet_ce(rao_alpha, rao_labels).mean()
    mode_edl = mean_modes_per_track(expected_dirichlet_ce(mode_alpha, mode_labels), observations, tracks)
    rao_kl = dirichlet_kl_to_uniform(suppress_true_class_evidence(rao_alpha, rao_labels)).mean()
    mode_kl = mean_modes_per_track(dirichlet_kl_to_uniform(suppress_true_class_evidence(mode_alpha, mode_labels)), observations, tracks)
    data_loss = (RAO_CLASSIFICATION_WEIGHT * rao_ce + MODE_CLASSIFICATION_WEIGHT * mode_ce
                 + RAO_EVIDENTIAL_WEIGHT * rao_edl + MODE_EVIDENTIAL_WEIGHT * mode_edl
                 + kl_coefficient * (rao_kl + mode_kl))
    return data_loss, {"rao_ce": rao_ce, "mode_ce": mode_ce, "rao_edl": rao_edl, "mode_edl": mode_edl,
                       "rao_kl": rao_kl, "mode_kl": mode_kl}


def model_forward():
    return model(X, edge_index, edge_types, observation_nodes, observation_track_index, len(series_ids))


def accuracy(logits, labels):
    return float((logits.argmax(-1) == labels).float().mean().detach().cpu())


@torch.inference_mode()
def all_split_metrics():
    model.eval()
    with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP and DEVICE.type == "cuda"):
        outputs = model_forward()
        metrics = {}
        for name, tracks in track_splits.items():
            observations = observation_splits[name]
            loss, parts = hierarchical_loss(outputs, name, EVIDENTIAL_KL_WEIGHT)
            metrics[name] = {"loss": float(loss.cpu()),
                             "track_rao_acc": accuracy(outputs["track_rao"][tracks], track_targets[tracks]),
                             "radar_mode_acc": accuracy(outputs["radar_mode"][observations], mode_targets[observations]),
                             **{key: float(value.cpu()) for key, value in parts.items()}}
    return metrics

EPOCHS = 200
ckpt_path = ARTIFACT_DIR / "best_track_model.pt"
writer = SummaryWriter(str(ARTIFACT_DIR / "tensorboard"))
history, best_val, bad_epochs = [], math.inf, 0
try:
    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        kl_coefficient = EVIDENTIAL_KL_WEIGHT * min(1.0, epoch / EVIDENTIAL_KL_ANNEAL_EPOCHS)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP and DEVICE.type == "cuda"):
            outputs = model_forward()
            data_loss, train_parts = hierarchical_loss(outputs, "train", kl_coefficient)
            regularization = L1_LAMBDA * sum(parameter.abs().sum() for parameter in model.parameters())
            loss = data_loss + regularization
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        row = {"epoch": epoch, "train_step_loss": float(data_loss.detach().cpu()), "kl_coefficient": kl_coefficient}
        metrics = all_split_metrics()
        for split_name, values in metrics.items():
            row.update({f"{split_name}_{key}": value for key, value in values.items()})
        history.append(row)
        for key, value in row.items():
            if key != "epoch": writer.add_scalar(key, value, epoch)
        if row["val_loss"] < best_val - EARLY_STOPPING_MIN_DELTA:
            best_val, bad_epochs = row["val_loss"], 0
            torch.save({"model_state": model.state_dict(), "epoch": epoch, "model_config": model_config,
                        "rao_vocab": rao_vocab, "mode_vocab": mode_vocab}, ckpt_path)
        else:
            bad_epochs += 1
        if epoch == 1 or epoch % 10 == 0:
            print(row)
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
finally:
    writer.close()


In [ ]:
# Load the hierarchical checkpoint selected by track-aware validation loss.
checkpoint = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
if checkpoint["rao_vocab"] != rao_vocab or checkpoint["mode_vocab"] != mode_vocab:
    raise ValueError("Checkpoint vocabularies do not match the current KG/data")
print(f"Loaded best track checkpoint from epoch {checkpoint['epoch']}: {ckpt_path}")


## Track-level outputs: one RAO assessment and a variable radar-mode sequence

The serialized operational unit is now a track. Each record contains exactly one KG-valid RAO prediction and evidential opinion, followed by ordered per-observation mode predictions and opinions. RAO is never decoded inside the observation loop, so its within-track violation count is zero by construction.


In [ ]:
final_metrics = all_split_metrics()
model.eval()
with torch.inference_mode(), torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP and DEVICE.type == "cuda"):
    final_outputs = model_forward()

# Deployment uses the one selected track RAO as a hard KG mask. Training above
# uses the soft marginal so gradients can flow through uncertain RAO alternatives.
selected_rao_classes = final_outputs["track_rao"].argmax(dim=-1)
observation_mode_mask = rao_mode_compatibility[selected_rao_classes[observation_track_index]].bool()
constrained_mode_logits = final_outputs["radar_mode"].masked_fill(~observation_mode_mask, torch.finfo(final_outputs["radar_mode"].dtype).min)

def ds_assessment(output, vocabulary, row_index, top_k=3):
    probabilities = output["probabilities"][row_index].detach().float().cpu().numpy()
    beliefs = output["belief"][row_index].detach().float().cpu().numpy()
    evidence = output["evidence"][row_index].detach().float().cpu().numpy()
    uncertainty = float(output["uncertainty"][row_index, 0].detach().float().cpu())
    order = np.argsort(probabilities)[::-1][:top_k]
    return {
        "full_frame_uncertainty": uncertainty,
        "dirichlet_strength": float(output["strength"][row_index, 0].detach().float().cpu()),
        "leading_hypotheses": [{"label": vocabulary[int(index)], "probability": float(probabilities[index]),
                                "belief": float(beliefs[index]), "plausibility": min(1.0, float(beliefs[index]) + uncertainty),
                                "evidence": float(evidence[index])} for index in order],
    }

track_predictions = []
for track_index, series_id in enumerate(series_ids):
    rao_class = int(final_outputs["track_rao"][track_index].argmax().detach().cpu())
    predicted_rao = dict(zip(RAO_FIELDS, rao_vocab[rao_class]))
    positions = torch.nonzero(observation_track_index == track_index, as_tuple=False).flatten().tolist()
    positions.sort(key=lambda position: node_meta[observation_node_indices[position]]["sequence_index"])
    mode_sequence = []
    for position in positions:
        node_idx = observation_node_indices[position]
        mode_class = int(constrained_mode_logits[position].argmax().detach().cpu())
        mode_sequence.append({
            "observation_id": node_meta[node_idx]["observation_id"],
            "sequence_index": node_meta[node_idx]["sequence_index"],
            "radar_mode": mode_vocab[mode_class],
            "mode_evidence": ds_assessment(final_outputs["evidential"]["radar_mode"], mode_vocab, position),
        })
    track_predictions.append({
        "series_id": series_id,
        "split": next(name for name, ids in split_series.items() if series_id in ids),
        "rao": predicted_rao,
        "rao_class_index": rao_class,
        "rao_evidence": ds_assessment(final_outputs["evidential"]["track_rao"], rao_vocab, track_index),
        "mode_sequence": mode_sequence,
        "rao_invariant": True,
    })

# Structural acceptance checks.
assert len(track_predictions) == len(series_ids)
assert all(tuple(record["rao"][field] for field in RAO_FIELDS) in rao_class_index for record in track_predictions)
assert all(record["rao_invariant"] for record in track_predictions)
assert all(observation_mode_mask[position, mode_class] for position, mode_class in enumerate(constrained_mode_logits.argmax(-1)))
RAO_VIOLATION_COUNT = 0
summary = {"model_config": model_config, "metrics": final_metrics, "rao_violation_count": RAO_VIOLATION_COUNT,
           "track_count": len(track_predictions), "observation_count": len(observation_node_indices)}
(ARTIFACT_DIR / "track_predictions.json").write_text(json.dumps(track_predictions, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
torch.save({"state_dict": model.state_dict(), "model_config": model_config, "rao_vocab": rao_vocab,
            "mode_vocab": mode_vocab}, ARTIFACT_DIR / "track_model_state_dict.pt")
print(json.dumps(summary, indent=2))
print(f"Saved {len(track_predictions):,} invariant track predictions")


In [ ]:
# Reload track-shaped artifacts for downstream dashboard and LLM cells.
summary = json.loads((ARTIFACT_DIR / "training_summary.json").read_text(encoding="utf-8"))
track_predictions = json.loads((ARTIFACT_DIR / "track_predictions.json").read_text(encoding="utf-8"))
print(f"Loaded {len(track_predictions):,} tracks; RAO violations={summary['rao_violation_count']}")


## Hierarchical evaluation dashboard

The dashboard separates the two prediction granularities. Its left side evaluates exact joint RAO classification once per held-out track, while its right side evaluates radar mode per observation and visualizes true/predicted mode changes. Calibration and uncertainty are likewise reported separately rather than mixing track and observation samples.


In [ ]:
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score

test_track_indices = track_splits["test"].detach().cpu().numpy()
test_observation_positions = observation_splits["test"].detach().cpu().numpy()
rao_probabilities = torch.softmax(final_outputs["track_rao"], -1).detach().float().cpu().numpy()
mode_probabilities = torch.softmax(constrained_mode_logits, -1).detach().float().cpu().numpy()
rao_truth = track_targets.detach().cpu().numpy()[test_track_indices]
rao_predicted = rao_probabilities[test_track_indices].argmax(1)
mode_truth = mode_targets.detach().cpu().numpy()[test_observation_positions]
mode_predicted = mode_probabilities[test_observation_positions].argmax(1)

def expected_calibration_error(probabilities, truth, bins=10):
    confidence, predicted = probabilities.max(1), probabilities.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    value = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        selected = (confidence > lower) & (confidence <= upper)
        if selected.any(): value += selected.mean() * abs((predicted[selected] == truth[selected]).mean() - confidence[selected].mean())
    return float(value)

def transition_labels(values, positions):
    result = []
    for left, right in zip(positions, positions[1:]): result.append(int(values[left] != values[right]))
    return result

true_transitions, predicted_transitions = [], []
for track_index in test_track_indices:
    positions = np.flatnonzero(observation_track_index.detach().cpu().numpy() == track_index)
    positions = positions[np.argsort([node_meta[observation_node_indices[p]]["sequence_index"] for p in positions])]
    true_transitions.extend(transition_labels(mode_targets.detach().cpu().numpy(), positions))
    predicted_transitions.extend(transition_labels(mode_probabilities.argmax(1), positions))
transition_f1 = f1_score(true_transitions, predicted_transitions, zero_division=0) if true_transitions else float("nan")

dashboard_metrics = {
    "track_rao_exact_accuracy": float((rao_truth == rao_predicted).mean()),
    "track_rao_macro_f1": float(f1_score(rao_truth, rao_predicted, average="macro", zero_division=0)),
    "track_rao_balanced_accuracy": float(balanced_accuracy_score(rao_truth, rao_predicted)),
    "track_rao_ece": expected_calibration_error(rao_probabilities[test_track_indices], rao_truth),
    "track_rao_mean_uncertainty": float(final_outputs["evidential"]["track_rao"]["uncertainty"][test_track_indices].mean().cpu()),
    "rao_violation_count": 0,
    "observation_mode_accuracy": float((mode_truth == mode_predicted).mean()),
    "observation_mode_macro_f1": float(f1_score(mode_truth, mode_predicted, average="macro", zero_division=0)),
    "observation_mode_ece": expected_calibration_error(mode_probabilities[test_observation_positions], mode_truth),
    "observation_mode_mean_uncertainty": float(final_outputs["evidential"]["radar_mode"]["uncertainty"][test_observation_positions].mean().cpu()),
    "mode_transition_f1": float(transition_f1),
}
(ARTIFACT_DIR / "hierarchical_dashboard_metrics.json").write_text(json.dumps(dashboard_metrics, indent=2), encoding="utf-8")
print(json.dumps(dashboard_metrics, indent=2))


In [ ]:
# Static dashboard: track RAO and observation-mode performance remain visibly separate.
rao_cm = confusion_matrix(rao_truth, rao_predicted, labels=np.unique(np.concatenate((rao_truth, rao_predicted))))
mode_cm = confusion_matrix(mode_truth, mode_predicted, labels=np.unique(np.concatenate((mode_truth, mode_predicted))))
fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
for axis, matrix, title in ((axes[0, 0], rao_cm, "Track-level joint RAO confusion"), (axes[0, 1], mode_cm, "Observation-level radar-mode confusion")):
    image = axis.imshow(matrix, cmap="Blues")
    axis.set(title=title, xlabel="Predicted class", ylabel="True class")
    fig.colorbar(image, ax=axis, fraction=.046)
metric_names = ["track_rao_exact_accuracy", "track_rao_macro_f1", "observation_mode_accuracy", "observation_mode_macro_f1", "mode_transition_f1"]
axes[1, 0].barh(metric_names, [dashboard_metrics[name] for name in metric_names], color=["#0072B2", "#56B4E9", "#009E73", "#6CC4A1", "#E69F00"])
axes[1, 0].set(xlim=(0, 1), title="Hierarchical held-out metrics")
axes[1, 0].grid(axis="x", alpha=.25)
rao_uncertainty = final_outputs["evidential"]["track_rao"]["uncertainty"][test_track_indices].detach().cpu().numpy().ravel()
mode_uncertainty = final_outputs["evidential"]["radar_mode"][test_observation_positions].detach().cpu().numpy().ravel()
axes[1, 1].hist(rao_uncertainty, bins=15, alpha=.7, label="RAO / track", color="#0072B2")
axes[1, 1].hist(mode_uncertainty, bins=15, alpha=.6, label="Mode / observation", color="#E69F00")
axes[1, 1].set(title="Full-frame uncertainty by prediction level", xlabel="Uncertainty", ylabel="Count")
axes[1, 1].legend()
fig.suptitle(f"Invariant track identification dashboard — RAO violations: {RAO_VIOLATION_COUNT}", fontsize=18, fontweight="bold")
dashboard_path = ARTIFACT_DIR / "track_identification_dashboard.png"
fig.savefig(dashboard_path, dpi=200, bbox_inches="tight")
plt.show()
print(dashboard_path)


In [ ]:
# Interactive mode timeline for a selected held-out track.
import plotly.graph_objects as go
DASHBOARD_TRACK_INDEX = int(os.environ.get("DASHBOARD_TRACK_INDEX", str(int(test_track_indices[0]))))
if DASHBOARD_TRACK_INDEX not in set(test_track_indices.tolist()):
    raise ValueError("DASHBOARD_TRACK_INDEX must identify a held-out test track")
positions = np.flatnonzero(observation_track_index.detach().cpu().numpy() == DASHBOARD_TRACK_INDEX)
positions = positions[np.argsort([node_meta[observation_node_indices[p]]["sequence_index"] for p in positions])]
true_modes = mode_targets.detach().cpu().numpy()[positions]
predicted_modes = mode_probabilities.argmax(1)[positions]
confidence = mode_probabilities[positions].max(1)
timeline = go.Figure()
timeline.add_trace(go.Scatter(x=np.arange(len(positions)), y=true_modes, mode="lines+markers", name="True radar mode"))
timeline.add_trace(go.Scatter(x=np.arange(len(positions)), y=predicted_modes, mode="lines+markers", name="Predicted radar mode",
                              marker={"size": 7 + 7 * confidence}, customdata=confidence,
                              hovertemplate="Observation %{x}<br>Mode class %{y}<br>Confidence %{customdata:.3f}<extra></extra>"))
predicted_rao = rao_vocab[int(rao_probabilities[DASHBOARD_TRACK_INDEX].argmax())]
timeline.update_layout(title=f"Track {series_ids[DASHBOARD_TRACK_INDEX]} — invariant RAO: {predicted_rao}",
                       xaxis_title="Ordered observation", yaxis_title="Radar-mode class", template="plotly_white")
timeline_path = ARTIFACT_DIR / "track_mode_timeline.html"
timeline.write_html(timeline_path, include_plotlyjs=True)
timeline.show()
print(timeline_path)


In [ ]:
# Display the track-shaped operational output used by downstream systems.
selected_track_output = track_predictions[DASHBOARD_TRACK_INDEX]
print(json.dumps({"series_id": selected_track_output["series_id"], "rao": selected_track_output["rao"],
                  "rao_invariant": selected_track_output["rao_invariant"],
                  "mode_sequence": [{"sequence_index": item["sequence_index"], "radar_mode": item["radar_mode"]}
                                    for item in selected_track_output["mode_sequence"]]}, indent=2))


In [ ]:
RAO_VIOLATION_COUNT


In [ ]:
summary


## Evidence-grounded LLM explanation for one complete track

Choose a track with `LLM_TRACK_INDEX` (default `0`). The prompt receives exactly one shared RAO assessment and the ordered, variable radar-mode sequence. It explicitly forbids inventing RAO changes between observations and asks the LLM to distinguish identity uncertainty from per-observation mode uncertainty. Held-out truth is attached only after generation.


In [ ]:
EXTREME_UNCERTAINTY = 0.60
HIGH_UNCERTAINTY = 0.35
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen3.5:9b")
OLLAMA_TIMEOUT_SECONDS = float(os.environ.get("OLLAMA_TIMEOUT_SECONDS", "120"))
OLLAMA_NUM_PREDICT = int(os.environ.get("OLLAMA_NUM_PREDICT", "384"))
LLM_TRACK_INDEX = int(os.environ.get("LLM_TRACK_INDEX", "0"))
if not -len(track_predictions) <= LLM_TRACK_INDEX < len(track_predictions):
    raise IndexError("LLM_TRACK_INDEX is outside the available track results")

def build_track_explanation_evidence(track_position):
    record = track_predictions[track_position]
    rao_uncertainty = float(record["rao_evidence"]["full_frame_uncertainty"])
    mode_uncertainties = [float(item["mode_evidence"]["full_frame_uncertainty"]) for item in record["mode_sequence"]]
    maximum_uncertainty = max([rao_uncertainty, *mode_uncertainties])
    return {
        "series_id": record["series_id"],
        "invariant_track_rao": record["rao"],
        "rao_evidence": record["rao_evidence"],
        "ordered_radar_mode_sequence": record["mode_sequence"],
        "model_constraint": "Aircraft, radar, and operator are one shared track variable; only radar mode may change by observation.",
        "evidence_assessment": {"maximum_uncertainty": maximum_uncertainty,
            "band": "extreme" if maximum_uncertainty >= EXTREME_UNCERTAINTY else "limited" if maximum_uncertainty >= HIGH_UNCERTAINTY else "supported",
            "active_collection_required": maximum_uncertainty >= EXTREME_UNCERTAINTY},
    }

def explanation_prompt(packet):
    return f"""You are an electronic-support analyst. Explain only the supplied model evidence for one complete track.
State the single invariant aircraft/radar/operator (RAO) identity once. Never imply that aircraft, radar, or operator changes between observations.
Then summarize the ordered radar-mode sequence and identify supported mode transitions; radar mode is the only attribute allowed to change.
Distinguish track-level RAO uncertainty from observation-level radar-mode uncertainty. Belief is committed singleton mass, plausibility includes full-frame ignorance, and pignistic probability is a decision probability rather than new evidence.
Do not invent facts. If the evidence band is extreme, state that the result is not decision-grade and recommend additional time-linked ESM and corroborating intelligence, including active radar collection when operationally appropriate.
Return a concise paragraph followed by RAO identity, Mode timeline, Uncertainty, and Recommended action bullets.

TRACK EVIDENCE JSON:
{json.dumps(packet, indent=2)}"""

def ollama_generate(prompt):
    parsed = urllib.parse.urlsplit(OLLAMA_BASE_URL)
    if parsed.scheme not in {"http", "https"} or parsed.hostname not in {"localhost", "127.0.0.1", "::1"}:
        raise ValueError("OLLAMA_BASE_URL must target a loopback-only Ollama instance")
    payload = json.dumps({"model": OLLAMA_MODEL, "prompt": prompt, "think": False, "stream": False,
                          "keep_alive": "10m", "options": {"temperature": 0.1, "num_predict": OLLAMA_NUM_PREDICT}}).encode()
    request = urllib.request.Request(f"{OLLAMA_BASE_URL}/api/generate", data=payload,
                                     headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(request, timeout=OLLAMA_TIMEOUT_SECONDS) as response:
            result = json.loads(response.read().decode())
    except urllib.error.URLError as error:
        raise RuntimeError(f"Unable to call local Ollama at {OLLAMA_BASE_URL}; start Ollama and pull {OLLAMA_MODEL!r}") from error
    explanation = result.get("response")
    if not isinstance(explanation, str) or not explanation.strip():
        raise RuntimeError(f"Ollama returned no explanation: {result}")
    return explanation.strip()

def explain_track(track_position):
    packet = build_track_explanation_evidence(track_position)
    prompt = explanation_prompt(packet)
    explanation = ollama_generate(prompt)
    if packet["evidence_assessment"]["active_collection_required"]:
        required = "Identification is not decision-grade; obtain additional time-linked ESM and corroborating intelligence, including active radar collection when operationally appropriate."
        if required not in explanation: explanation += f"\n\nRecommended action: {required}"
    return {"series_id": packet["series_id"], "correct_track_identification": track_targets_by_series_id[packet["series_id"]],
            "ollama_model": OLLAMA_MODEL, "evidence": packet, "prompt": prompt, "explanation": explanation}

track_explanation = explain_track(LLM_TRACK_INDEX)
explanation_path = ARTIFACT_DIR / f"llm_track_explanation_{LLM_TRACK_INDEX}.json"
explanation_path.write_text(json.dumps(track_explanation, indent=2), encoding="utf-8")
print(track_explanation["explanation"])
print("\nCorrect held-out track identification (not supplied to the LLM):")
print(json.dumps(track_explanation["correct_track_identification"], indent=2))
print(explanation_path)


In [ ]:
# Training curves for the two prediction levels.
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for split in ("train", "test", "val"):
    axes[0].plot(epochs, [row[f"{split}_track_rao_acc"] for row in history], label=split)
    axes[1].plot(epochs, [row[f"{split}_radar_mode_acc"] for row in history], label=split)
axes[0].set(title="Track-level joint RAO accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
axes[1].set(title="Observation-level radar-mode accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
for axis in axes: axis.legend(); axis.grid(True, alpha=.3)
fig.tight_layout()
training_plot_path = ARTIFACT_DIR / "hierarchical_training_metrics.png"
fig.savefig(training_plot_path, dpi=150)
training_plot_path
